# Module 12 Lab - Ethics, Fairness, and Bias in ML

**Objective:** To understand how machine learning models can inherit and amplify societal biases, how to measure this bias using fairness metrics, and to think critically about the ethical implications of deploying ML systems.

**In this lab, you will train a model on a real-world dataset and audit it for fairness across different demographic groups.**

## Part 1: What is Algorithmic Bias?

**Concept:** Machine learning models learn from data. If the data reflects existing societal biases, the model will learn those biases. An "unbiased" algorithm trained on biased data will produce a biased model. This can lead to systems that are systematically unfair to certain groups of people.

**Sources of Bias:**
*   **Historical Bias:** The data reflects a world with historical injustices (e.g., past hiring data may show fewer women in leadership roles).
*   **Measurement Bias:** The way we collect or measure data is flawed (e.g., using arrest records as a proxy for crime, which can be influenced by policing patterns).
*   **Representation Bias:** The data underrepresents certain groups, so the model doesn't learn to perform well for them.

**Problem:** We will use the "Adult" dataset, which is used to predict whether an individual's income is greater than $50k/year. It contains sensitive attributes like `sex` and `race`, which we can use to audit our model for bias.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

# Load the data
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
df = pd.read_csv(url, header=None, names=columns, sep=',\s*', engine='python', na_values='?')

# Data Cleaning
df.dropna(inplace=True)
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})

X = df.drop('income', axis=1)
y = df['income']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Create a preprocessing pipeline
numeric_features = X.select_dtypes(include='number').columns
categorical_features = X.select_dtypes(exclude='number').columns

preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)
)

# Train a baseline model
model = make_pipeline(preprocessor, LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

print(f"Overall model accuracy: {model.score(X_test, y_test):.2%}")

<>:12: SyntaxWarning: invalid escape sequence '\s'
<>:12: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_5266/2547676487.py:12: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(url, header=None, names=columns, sep=',\s*', engine='python', na_values='?')


Overall model accuracy: 84.61%


## Part 2: Auditing the Model for Fairness

High overall accuracy can hide poor performance on specific subgroups. We need to audit the model by comparing its performance across sensitive attributes like `sex`.

**Concept: Group Fairness**
One common fairness goal is to ensure the model works equally well for different groups. We can measure this by calculating metrics for each group separately.

**Your Task:** Create a function to calculate accuracy for different subgroups and then use it to compare the model's performance for males and females.

In [2]:
def get_subgroup_accuracy(model, X_test, y_test, subgroup_column, subgroup_value):
    """Calculates accuracy for a specific subgroup of the test data."""
    # 1. Create a boolean mask to select the subgroup from X_test
    subgroup_mask = X_test[subgroup_column] == subgroup_value

    # 2. Select the subgroup data
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    # 3. Calculate and return the model's score on this subgroup
    return model.score(X_subgroup, y_subgroup)

# 4. Calculate accuracy for males and females
acc_male = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Male')
acc_female = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Female')

print(f"Accuracy for Males: {acc_male:.2%}")
print(f"Accuracy for Females: {acc_female:.2%}")

Accuracy for Males: 81.20%
Accuracy for Females: 91.81%


### Task 2: Deeper Dive with a Confusion Matrix

Accuracy alone doesn't tell the whole story. Let's look at the types of errors the model makes for each group.

**Your Task:** Calculate and compare the **False Positive Rate (FPR)** and **False Negative Rate (FNR)** for males and females.

*   **FPR:** `FP / (FP + TN)` - The percentage of people who did NOT have high income but were incorrectly predicted to have high income.
*   **FNR:** `FN / (FN + TP)` - The percentage of people who DID have high income but were incorrectly predicted to have low income.

In [3]:
from sklearn.metrics import confusion_matrix

def get_rates(model, X_test, y_test, subgroup_column, subgroup_value):
    subgroup_mask = X_test[subgroup_column] == subgroup_value
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    y_pred_subgroup = model.predict(X_subgroup)
    tn, fp, fn, tp = confusion_matrix(y_subgroup, y_pred_subgroup).ravel()

    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)
    return fpr, fnr

# 1. Calculate the rates for males and females
fpr_male, fnr_male = get_rates(model, X_test, y_test, 'sex', 'Male')
fpr_female, fnr_female = get_rates(model, X_test, y_test, 'sex', 'Female')

print(f"Male   - False Positive Rate: {fpr_male:.2%}, False Negative Rate: {fnr_male:.2%}")
print(f"Female - False Positive Rate: {fpr_female:.2%}, False Negative Rate: {fnr_female:.2%}")

Male   - False Positive Rate: 10.26%, False Negative Rate: 37.80%
Female - False Positive Rate: 2.81%, False Negative Rate: 47.84%


## Reflective Knowledge Check

**Instructions:** Answer the following questions based on the results from the code above.

---

**1. Analyze Your Results:** Look at the subgroup accuracies you calculated. Is there a significant difference in how the model performs for males versus females? Which group does the model perform better for?

Yes, there is a significant difference. The model performs much better for females (91.81%) than for males (81.20%) — a gap of over 10 percentage points. This is a surprising result because it shows that high accuracy for one group does not mean the model is fair overall. The model appears more accurate for females largely because most females in this dataset earn ≤$50K, making it easy to correctly classify them. The model struggles more with males, who have a more diverse distribution of income levels. This highlights why we must look beyond overall accuracy and examine subgroup performance carefully.

---

**2. Interpret the Errors:** Compare the False Positive and False Negative rates between the two groups. For which group is the model more likely to make a False Positive error? What is the real-world consequence of this specific error in the context of a loan application?

The model is more likely to make a False Positive error for males (FPR: 10.26%) than for females (FPR: 2.81%). A False Positive means the model predicts someone will have high income when they actually do not. In the context of a loan application, a False Positive would mean the model incorrectly approves a loan for someone who cannot actually afford to repay it. This can cause financial harm to the borrower — they take on debt they cannot repay, leading to default and damaged credit.

However, the more harmful disparity is in the False Negative Rate. Females have an FNR of 47.84% compared to 37.80% for males. This means nearly half of all high-income women are being incorrectly classified as low-income and would be denied loans they actually qualify for — a clear and harmful systematic bias against women.

---

**3. Justify a Decision:** Would you approve this model for deployment in a hiring process? Justify your decision.

I would NOT approve this model for deployment in a hiring process. The False Negative Rate for females is 47.84%, compared to 37.80% for males. In a hiring context, a False Negative means a qualified candidate is incorrectly screened out. This means nearly half of all high-income women who are qualified for the role would be unfairly rejected by the model, while only about 38% of equally qualified men face the same fate.

This represents a clear disparate impact on female candidates that could violate equal employment opportunity laws. The FNR is the most critical error type in hiring because it directly harms qualified individuals by denying them opportunities they deserve. Even though the model shows higher overall accuracy for females (91.81%), this is misleading — the high accuracy is driven by correctly predicting low-income females, not by correctly identifying high-income ones. Until this bias is addressed through fairness-aware techniques, this model should not be used in hiring.

---

**4. Propose a Mitigation:** If you removed the 'sex' column and retrained the model, would it become fair? Why or why not?

Simply removing the 'sex' column would likely NOT make the model fair. This approach is called 'fairness through unawareness,' and research has consistently shown that it fails in practice. Many other features in the dataset are highly correlated with sex. For example, 'occupation', 'marital-status', 'relationship', and 'hours-per-week' all carry implicit gender information due to historical and societal patterns. Women are historically overrepresented in certain lower-paying occupations and underrepresented in others, and marital status categories like 'Wife' directly encode gender. Even without the 'sex' column, the model can reconstruct a proxy for gender from these correlated features and continue making biased predictions. True fairness mitigation requires more sophisticated approaches such as re-weighting training samples, adversarial debiasing, post-processing outputs to equalize error rates, or using fairness-aware algorithms that explicitly optimize for equalized odds or demographic parity.